# 05b — Fluid Bed Coating Digital Twin

Full process chain: **Pre-heating → Spraying → Drying → Dissolution prediction**.

**r_spraying** and **r_drying** are computed automatically from process parameters using the
AICc-selected empirical correlations fitted on the 19-run Bosch 2018 DoE (Steps 06a/06b).

* **r_spray** model: spray rate + coating conc + DM ratio + coating conc × DM ratio  (R²=0.844)
* **r_dry** model: batch size + DM ratio + SSA + humidity + DM ratio × humidity  (R²=0.851)

The dotted line shows the theoretical maximum WG assuming **no coating loss**.
Slide **Sample time** to withdraw a virtual sample and see the predicted dissolution profile.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.config import DISSOLUTION
from fluid_bed.simulate import run_full_process
from fluid_bed.models.dissolution import dissolution_curve

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

In [ ]:
# Physical constants, coating-loss correlations, the PH→SP→DR ODE chain and the
# dissolution model all live in the fluid_bed package (config.py, simulate.py,
# models/dissolution.py) — this cell is UI only.

_sim_cache = {}

# ── Sliders ───────────────────────────────────────────────────────────────────
S  = dict(continuous_update=False)
_DW = {'description_width': '170px'}

_w = {
    # Particle / batch
    'd_mm':          widgets.FloatSlider(1.00, min=0.80, max=1.50, step=0.05, description='Diameter (mm)',      style=_DW, **S),
    'ssa_cm2g':      widgets.FloatSlider(65.0, min=55.0, max=75.0, step=1.0,  description='SSA (cm2/g)',        style=_DW, **S),
    'T0_C':          widgets.FloatSlider(20.0, min=15.0, max=30.0, step=1.0,  description='T0 particle (C)',    style=_DW, **S),
    'batch_kg':      widgets.FloatSlider(4.6,  min=3.0,  max=6.0,  step=0.1,  description='Batch size (kg)',    style=_DW, **S),
    'humidity_g_kg': widgets.FloatSlider(13.4, min=0.0,  max=25.0, step=0.5,  description='Humidity (g/kg)',    style=_DW, **S),
    'dmc_pct':       widgets.FloatSlider(1.5,  min=1.0,  max=2.0,  step=0.1,  description='DMC (% m/m)',        style=_DW, **S),
    'coating_level': widgets.FloatSlider(0.5,  min=-1.0, max=1.0,  step=0.05, description='Coating level',      style=_DW, **S),
    # Pre-heating
    'ph_T_C':        widgets.FloatSlider(50.0, min=40.0, max=60.0, step=2.0,  description='[PH] T inlet (C)',   style=_DW, **S),
    'ph_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[PH] Flow (m3/h)',   style=_DW, **S),
    'ph_dur_min':    widgets.FloatSlider(20.0, min=10.0, max=30.0, step=1.0,  description='[PH] Dur (min)',     style=_DW, **S),
    # Spraying
    'sp_T_C':        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,  description='[SP] T inlet (C)',   style=_DW, **S),
    'sp_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[SP] Flow (m3/h)',   style=_DW, **S),
    'sp_rate_g_min': widgets.FloatSlider(120,  min=95,   max=150,  step=5,    description='[SP] Spray (g/min)', style=_DW, **S),
    # Drying
    'dr_T_C':        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,  description='[DR] T inlet (C)',   style=_DW, **S),
    'dr_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[DR] Flow (m3/h)',   style=_DW, **S),
    'dr_dur_min':    widgets.FloatSlider(9.0,  min=3.0,  max=15.0, step=1.0,  description='[DR] Dur (min)',     style=_DW, **S),
}

_w_t = {'t_proc_min': widgets.FloatSlider(
    30.0, min=0.0, max=90.0, step=0.5, description='Sample time (min)',
    style=_DW, **S, layout=widgets.Layout(width='500px'))}

# ── Panel layout ──────────────────────────────────────────────────────────────
def _lbl(text, color):
    return widgets.HTML(
        f'<b style="background:{color};color:white;padding:3px 10px;'
        f'border-radius:4px;display:inline-block;margin:4px 0">{text}</b>')

_left_col = widgets.VBox([
    _lbl('Particle / Batch', '#5B2C6F'),
    _w['d_mm'], _w['ssa_cm2g'], _w['T0_C'],
    _w['batch_kg'], _w['humidity_g_kg'], _w['dmc_pct'], _w['coating_level'],
])
_right_col_1 = widgets.VBox([
    _lbl('Pre-heating', '#4472C4'),
    _w['ph_T_C'], _w['ph_flow'], _w['ph_dur_min'],
])
_right_col_2 = widgets.VBox([
    _lbl('Spraying  [r_spray from correlation]', '#ED7D31'),
    _w['sp_T_C'], _w['sp_flow'], _w['sp_rate_g_min'],
])
_right_col_3 = widgets.VBox([
    _lbl('Drying  [r_dry from correlation]', '#1E8449'),
    _w['dr_T_C'], _w['dr_flow'], _w['dr_dur_min'],
])
_panel = widgets.HBox([_left_col, _right_col_1, _right_col_2, _right_col_3],
                      layout=widgets.Layout(gap='30px'))

# ── Stage helpers ─────────────────────────────────────────────────────────────
_SC = {'PH': 'royalblue', 'SP': 'darkorange', 'DR': 'seagreen'}

def _stage_at(t, ph_end, sp_end):
    if t <= ph_end: return 'PH', _SC['PH']
    if t <= sp_end: return 'SP', _SC['SP']
    return 'DR', _SC['DR']

def _shade(ax, ph_end, sp_end, t_end):
    ax.axvspan(0, ph_end, alpha=0.07, color=_SC['PH'])
    ax.axvspan(ph_end, sp_end, alpha=0.07, color=_SC['SP'])
    ax.axvspan(sp_end, t_end, alpha=0.07, color=_SC['DR'])
    ax.axvline(ph_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.axvline(sp_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.set_xlabel('Time (min)')

def _stxt(ax, ph_end, sp_end, t_end):
    lo, hi = ax.get_ylim(); yp = lo + (hi - lo) * 0.97
    for pos, lbl in [((0+ph_end)/2, 'PH'),
                     ((ph_end+sp_end)/2, 'SP'),
                     ((sp_end+t_end)/2, 'DR')]:
        ax.text(pos, yp, lbl, ha='center', va='top', fontsize=8,
                color=_SC[lbl], fontweight='bold')

# ── Process simulation ────────────────────────────────────────────────────────
def _run05(d_mm, ssa_cm2g, T0_C, batch_kg, humidity_g_kg, dmc_pct, coating_level,
           ph_T_C, ph_flow, ph_dur_min, sp_T_C, sp_flow, sp_rate_g_min,
           dr_T_C, dr_flow, dr_dur_min):

    sim = run_full_process(
        d_mm, ssa_cm2g, T0_C, batch_kg, humidity_g_kg, dmc_pct, coating_level,
        ph_T_C, ph_flow, ph_dur_min, sp_T_C, sp_flow, sp_rate_g_min,
        dr_T_C, dr_flow, dr_dur_min)

    t_all = sim.t_all
    ph_end, sp_end, t_end = sim.ph_end, sim.sp_end, sim.t_end

    if _w_t['t_proc_min'].value > t_end:
        _w_t['t_proc_min'].value = t_end
    _w_t['t_proc_min'].max = t_end

    _sim_cache.update(dict(sim=sim, ssa_cm2g=ssa_cm2g))

    t_step = [0, ph_end, ph_end, sp_end, sp_end, t_end]
    T_step = [ph_T_C, ph_T_C, sp_T_C, sp_T_C, dr_T_C, dr_T_C]

    fig, axes = plt.subplots(2, 2, figsize=(12, 5))
    fig.suptitle(
        f'Fluid Bed Coating — Empirical correlations  |  '
        f'r_spray = {sim.r_spraying*1e6:.1f} x1e-6 kg/s  |  r_dry = {sim.r_drying*1e3:.2f} x1e-3 kg/s  |  '
        f'DM ratio = {sim.dm_ratio_g_kg:.2f} g/kg',
        fontsize=11, fontweight='bold')

    ax = axes[0, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.step(t_step, T_step, color='tomato', lw=1.5, ls='--', where='post', label='T inlet')
    ax.plot(t_all, sim.T_gas, color='tomato', lw=1.5, alpha=0.5, label='T outlet')
    ax.plot(t_all, sim.T_product, color='steelblue', lw=2.5, label='T product')
    ax.set_ylabel('Temperature (C)'); ax.set_title('Temperature profiles')
    ax.legend(fontsize=8, loc='lower left'); ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[0, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, sim.Y_particle, color='darkorange', lw=2.5)
    ax.set_ylabel('Acetone on particles (wt %)'); ax.set_title('Particle acetone')
    ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[1, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, sim.Y_gas, color='cadetblue', lw=2.5)
    ax.set_ylabel('Acetone in gas (wt %)'); ax.set_title('Gas-phase acetone')
    ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[1, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, sim.WG_noloss, color='mediumpurple', lw=1.5, ls=':', alpha=0.8, label='WG no loss')
    ax.plot(t_all, sim.WG, color='mediumpurple', lw=2.5, label='WG model')
    ax.set_ylabel('Coating weight gain (%)'); ax.set_title('Coating WG')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    plt.tight_layout(); plt.show(); plt.close()

    print(f'Spray: {sim.qty_sol_kg*1000:.0f} g in {sim.sp_dur_s/60:.1f} min  |  Total: {t_end:.1f} min  '
          f'|  DM ratio: {sim.dm_ratio_g_kg:.2f} g/kg  |  humidity: {humidity_g_kg:.1f} g/kg')
    print(f'r_spray = {sim.r_spraying*1e6:.2f} x1e-6 kg/s  |  r_dry = {sim.r_drying*1e3:.3f} x1e-3 kg/s')
    print(f'WG end-spray: {sim.wg_end_spray:.3f}%  |  WG final: {sim.wg_final:.3f}%  |  WG no-loss: {sim.wg_max_noloss:.3f}%')
    print('Move the Sample time slider below to update the dissolution profile.')


# ── Dissolution panel ─────────────────────────────────────────────────────────
def _run_disso(t_proc_min):
    if not _sim_cache:
        print('Move any process slider to run the simulation first.')
        return

    sim = _sim_cache['sim']
    ssa = _sim_cache['ssa_cm2g']
    t_all, WG, WG_noloss = sim.t_all, sim.WG, sim.WG_noloss
    ph_end, sp_end, t_end = sim.ph_end, sim.sp_end, sim.t_end

    t_s        = float(np.clip(t_proc_min, 0.0, t_end))
    wg_at_t    = float(np.interp(t_s, t_all, WG))
    wg_nl_at_t = float(np.interp(t_s, t_all, WG_noloss))
    stage_name, stage_color = _stage_at(t_s, ph_end, sp_end)

    t_diss, F_diss, k    = dissolution_curve(wg_at_t / 100.0, ssa)
    _,      F_noloss, _  = dissolution_curve(wg_nl_at_t / 100.0, ssa)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(
        f'Virtual Sample — t = {t_s:.1f} min  [{stage_name}]  '
        f'WG = {wg_at_t:.3f}%  (no-loss: {wg_nl_at_t:.3f}%)   k = {k:.4e} s-1',
        fontsize=11, fontweight='bold')

    # Left: WG profile + sample marker
    ax = axes[0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, WG_noloss, color='mediumpurple', lw=1.5, ls=':', alpha=0.8, label='WG no loss')
    ax.plot(t_all, WG, color='mediumpurple', lw=2.0, label='WG model')
    bw = max(t_end * 0.008, 0.3)
    ax.axvspan(t_s - bw, t_s + bw, alpha=0.55, color=stage_color, label=f'Sample [{stage_name}]')
    ax.axvline(t_s, color=stage_color, lw=1.5)
    ax.axhline(wg_at_t, color=stage_color, lw=1.0, ls=':', alpha=0.6)
    ax.annotate(f'{wg_at_t:.3f}%', xy=(t_s, wg_at_t),
                xytext=(t_s + t_end*0.04, wg_at_t), fontsize=9, color=stage_color, va='center')
    _stxt(ax, ph_end, sp_end, t_end)
    ax.set_xlabel('Process time (min)'); ax.set_ylabel('Coating WG (%)')
    ax.set_title('Coating evolution — sample point')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Right: dissolution curve (model vs no-loss)
    ax = axes[1]
    ax.set_facecolor(to_rgba(stage_color, 0.06))
    ax.plot(t_diss, F_noloss, color='dimgrey', lw=1.5, ls=':', alpha=0.8, label='No loss')
    ax.plot(t_diss, F_diss, color=stage_color, lw=2.5, label='Model')
    for tgt, ls in [(50, '--'), (80, ':')]:
        if F_diss[-1] >= tgt:
            t_m = float(np.interp(tgt, F_diss, t_diss))
            ax.axhline(tgt, color='grey', lw=0.8, ls=ls, alpha=0.5)
            ax.axvline(t_m, color='grey', lw=0.8, ls=ls, alpha=0.5)
            ax.text(t_m + 3, tgt + 1.5, f't{tgt} = {t_m:.0f} min', fontsize=8, color='dimgrey')
    ax.set_xlim(0, DISSOLUTION['Total_min']); ax.set_ylim(0, 105)
    ax.set_xlabel('Dissolution time (min)'); ax.set_ylabel('Drug released (%)')
    ax.set_title(f'First-order dissolution  [{stage_name}]')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show(); plt.close()

    t50 = float(np.interp(50, F_diss, t_diss)) if F_diss[-1] >= 50 else float('nan')
    t80 = float(np.interp(80, F_diss, t_diss)) if F_diss[-1] >= 80 else float('nan')
    t50_s = f'{t50:.0f} min' if not np.isnan(t50) else '> 240 min'
    t80_s = f'{t80:.0f} min' if not np.isnan(t80) else '> 240 min'
    print(f't50 = {t50_s}  |  t80 = {t80_s}  |  k = {k:.4e} s-1  |  stage = {stage_name}')


# ── Wire up and display ───────────────────────────────────────────────────────
_out_proc  = widgets.interactive_output(_run05, _w)
_out_disso = widgets.interactive_output(_run_disso, _w_t)

_disso_header = widgets.VBox([
    widgets.HTML(
        '<b style="background:#8E44AD;color:white;padding:3px 10px;'
        'border-radius:4px;display:inline-block;margin:8px 0">'
        'Dissolution — Virtual Sample Withdrawal</b>'
        '<br><small>Slide to any process time. '
        'Slider max auto-adjusts to the actual total process duration.</small>'
    ),
    _w_t['t_proc_min'],
])

clear_output(wait=True)
display(_panel, _out_proc, _disso_header, _out_disso)